# 01 - pandas Basics 🐼🐧

**pandas** is Python's spreadsheet superpower. Its main object is the **DataFrame**, a table with named columns (like a spreadsheet or a CSV file) where each column is basically a NumPy array.

Today's dataset: **Palmer Penguins**, measurements of 344 penguins from three islands near Palmer Station, Antarctica, collected by Dr. Kristen Gorman. (The data is public domain, CC0.)

| Column | Meaning |
|--------|---------|
| `species` | Adelie, Chinstrap, or Gentoo |
| `island` | Biscoe, Dream, or Torgersen |
| `bill_length_mm`, `bill_depth_mm` | beak size |
| `flipper_length_mm` | wing length |
| `body_mass_g` | weight in grams |
| `sex` | MALE / FEMALE (sometimes missing) |

In [ ]:
import pandas as pd

penguins = pd.read_csv("data/penguins.csv")
penguins.head()          # first 5 rows

## 1. First look at the data
Always start by asking: How big is it? What are the columns? What types? Anything missing?

In [ ]:
print(penguins.shape)          # (rows, columns)
print(penguins.columns.tolist())

In [ ]:
penguins.info()                # column types and non-missing counts

In [ ]:
penguins.describe()            # summary statistics for the numeric columns

In [ ]:
penguins.isna().sum()          # how many missing values in each column?

## 2. Selecting columns
- One column → a **Series** (like a labeled 1D array): `df["col"]`
- Several columns → a **DataFrame**: `df[["col1", "col2"]]` (note the double brackets!)

In [ ]:
mass = penguins["body_mass_g"]
print(type(mass))
print("Average mass:", round(mass.mean()), "grams")
print("Heaviest:", mass.max(), " Lightest:", mass.min())

In [ ]:
penguins[["species", "island", "body_mass_g"]].head(3)

In [ ]:
penguins["species"].value_counts()      # count each category

In [ ]:
penguins["island"].unique()

## 3. Filtering rows with boolean masks
This is the same idea as NumPy masks from Day 7. Combine conditions with `&` (and), `|` (or), and `~` (not), and **put parentheses around each condition**.

In [ ]:
heavy = penguins[penguins["body_mass_g"] > 5500]
print(len(heavy), "penguins over 5.5 kg")
heavy.head()

In [ ]:
# Gentoo penguins on Biscoe island with flippers over 220 mm
mask = (penguins["species"] == "Gentoo") & (penguins["flipper_length_mm"] > 220)
penguins[mask].head()

In [ ]:
# .isin() checks membership in a list
penguins[penguins["island"].isin(["Dream", "Torgersen"])]["species"].value_counts()

## 4. Sorting, and picking rows by position or label

In [ ]:
penguins.sort_values("body_mass_g", ascending=False).head(5)

In [ ]:
print(penguins.iloc[0])            # row by POSITION (like list indexing)
print()
print(penguins.loc[5, "species"])  # by LABEL: row label 5, column "species"

## 5. Missing values
Real data is messy. We can **drop** rows with missing values, or **fill** them in.

In [ ]:
clean = penguins.dropna()
print(len(penguins), "->", len(clean), "rows after dropna()")

## 6. Creating new columns
Column math is vectorized, just like NumPy.

In [ ]:
clean = clean.copy()
clean["body_mass_kg"] = clean["body_mass_g"] / 1000
clean["bill_ratio"] = clean["bill_length_mm"] / clean["bill_depth_mm"]
clean["size"] = clean["body_mass_g"].apply(lambda g: "big" if g >= 4500 else "small")   # apply any function!
clean[["species", "body_mass_kg", "bill_ratio", "size"]].head()

## 7. groupby: split → apply → combine
"For each species, what's the average ___?" This is one of the most useful operations in all of data analysis.

In [ ]:
clean.groupby("species")["body_mass_g"].mean().round()

In [ ]:
clean.groupby("species")[["bill_length_mm", "flipper_length_mm", "body_mass_g"]].mean().round(1)

In [ ]:
# Group by TWO columns
clean.groupby(["species", "sex"])["body_mass_g"].mean().round()

In [ ]:
# Several statistics at once
clean.groupby("island")["body_mass_g"].agg(["count", "mean", "min", "max"]).round()

## 8. Saving

In [ ]:
import os
os.makedirs("output", exist_ok=True)
summary = clean.groupby("species")[["body_mass_g", "flipper_length_mm"]].mean().round(1)
summary.to_csv("output/penguin_summary.csv")
summary

## ✏️ Try it
1. Which island has the **most** penguins?
2. What's the average **flipper length** of female Adelie penguins?
3. Which species has the longest bills on average?
4. How many penguins weigh between 3,000 and 3,500 grams?

In [ ]:
# your code here

## Cheat sheet
| Task | Code |
|------|------|
| Load CSV | `pd.read_csv("file.csv")` |
| Peek | `df.head()`, `df.tail()`, `df.shape`, `df.info()`, `df.describe()` |
| Columns | `df["a"]`, `df[["a", "b"]]` |
| Filter | `df[df["a"] > 5]`, `df[(cond1) & (cond2)]` |
| Count categories | `df["a"].value_counts()` |
| Sort | `df.sort_values("a", ascending=False)` |
| Missing | `df.isna().sum()`, `df.dropna()`, `df.fillna(0)` |
| New column | `df["c"] = df["a"] / df["b"]`, `df["a"].apply(func)` |
| Group | `df.groupby("a")["b"].mean()` |
| Save | `df.to_csv("out.csv")` |